# Data Health Audit & Preparation

## Important Note

This notebook `/notebooks/3_data_health_audit_&_prep_global.ipynb` is different than the sub-sampled version `/notebooks/3_data_health_audit_&_prep_only.ipynb` in terms of structure and purpose. 

Purpose: The sub-sampled version aims to check the data health and explore potential issue in various aspect of the dataset. This one aims to verify if the issues/trend/patterns found in the sub-sampled counterpart will be similar to this version. The similar pattern exhibits from two notebooks will indirectly indicates that the subsampling strategy is unbiased and effective, and vice versa.

Structure: The subsampled version will explicitly write down the logic as you audit the data health, whereas this version will put most workflows in `/src/utils/data_health_audit_helper.py`, and the audit logic will call a specific function to perform and verify health check. 

In [1]:
# Make sure you are at the parent directory
from pathlib import Path
import sys

# Define MODE
MODE = "EXPANSE" # "EXPANSE" is the only mode
if MODE.upper() != "EXPANSE":
    raise Exception("Invalid mode, the only acceptible is 'EXPANSE'")

# Root path by MODE
PROJECT_ROOT_BY_MODE = {
    "COLAB": Path("/content/drive/MyDrive/DSC 288R/Project"),
    "EXPANSE": Path("/home/bguo3/bguo3/DSC-288R-Capstone-Final-Project"),
    "LOCAL": Path("/Users/steveg/Desktop/DSC-288R-Capstone-Final-Project"),
}
ROOT = PROJECT_ROOT_BY_MODE[MODE.upper()]

# Add the root path to global system
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Print the root path
print("MODE:", MODE)
print("PROJECT_ROOT:", ROOT)

MODE: EXPANSE
PROJECT_ROOT: /home/bguo3/bguo3/DSC-288R-Capstone-Final-Project


## Import Modules

In [2]:
import os

from src.utils.pyspark_utils import create_spark_session, memory_count
from src.utils.paths_utils import ProjectPaths
from src.utils.io_utils import read_spark_parquet, write_pandas_parquet
from src.pipelines.data_health_audit import print_section, format_report
import src.pipelines.data_health_audit as audit
import src.pipelines.data_prep as prep

# Data health audit functions under audit includes:
    # structure_report
    # schema_audit
    # null_report
    # consistency_report
    # validity_report
    # anomaly_report
    # noise_report
    # uniqueness_report
    # duplicate_report
    
    # print_section
    # format_report

Matplotlib created a temporary cache directory at /scratch/bguo3/job_49054060/matplotlib-6xh697bl because the default path (/home/jovyan/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


## Set up for Spark

## Read Full Data Parquet File

In [3]:
# # Set up a dedicated job-running spill storage directory
# SPARK_LOCAL_DIR = f"/scratch/{os.environ['USER']}/job_{os.environ['SLURM_JOB_ID']}/spark-local"
# os.makedirs(SPARK_LOCAL_DIR, exist_ok=True)

# Set up for Spark app & resource allocation
spark = create_spark_session("steam_reviews_data_health_check")

# Load data
paths = ProjectPaths(MODE)
raw_df = read_spark_parquet(spark, paths.full_parquet)

Read Spark parquet from: /expanse/lustre/scratch/bguo3/temp_project/steam_reviews/full_parquet


## Health Audit - Dataset Structure

In [6]:
ROW_COUNT = raw_df.count()

## Shape
# 1. (# of rows, # of columns, # of cells)
print_section("1. (# of rows, # of columns, # of cells)")
display(audit.structure_report(raw_df, ROW_COUNT))

## 2. Concise summary on columns details
print_section("2. Concise summary on columns details")
display(audit.schema_audit(raw_df))
raw_df.printSchema()

# 3. Memory count
print_section("3. Memory count")
memory_count(raw_df)


1. (# of rows, # of columns, # of cells)


,row_count,column_count,cell_count
0,113885601,18,2049940818



2. Concise summary on columns details


,column_name,present_in_df,expected_type,actual_type,matches_expected_family
0,author_steamid,True,bigint,bigint,True
1,appid,True,int,int,True
2,author_num_games_owned,True,int,int,True
3,author_num_reviews,True,int,int,True
4,author_playtime_forever,True,int,int,True
5,author_playtime_last_two_weeks,True,int,int,True
6,author_playtime_at_review,True,int,int,True
7,author_last_played,True,bigint,bigint,True
8,review,True,string,string,True
9,voted_up,True,boolean,boolean,True


root
 |-- author_steamid: long (nullable = true)
 |-- appid: integer (nullable = true)
 |-- author_num_games_owned: integer (nullable = true)
 |-- author_num_reviews: integer (nullable = true)
 |-- author_playtime_forever: integer (nullable = true)
 |-- author_playtime_last_two_weeks: integer (nullable = true)
 |-- author_playtime_at_review: integer (nullable = true)
 |-- author_last_played: long (nullable = true)
 |-- review: string (nullable = true)
 |-- voted_up: boolean (nullable = true)
 |-- votes_up: integer (nullable = true)
 |-- votes_funny: long (nullable = true)
 |-- weighted_vote_score: float (nullable = true)
 |-- comment_count: integer (nullable = true)
 |-- written_during_early_access: boolean (nullable = true)
 |-- timestamp_created: long (nullable = true)
 |-- timestamp_updated: long (nullable = true)
 |-- language: string (nullable = true)


3. Memory count
Total estimated size: 55.46 GB


## Data Preparation - Enforce Schema

In [7]:
df = raw_df.transform(prep.enforce_schema)

## Health Audit - Completeness

In [ ]:
## Compeleteness helps us investigate missing value and if we have useful
## features for prediction problems

# 1. Identify missing values per column
# 2. Calculate percentage of missing values
audit.null_report(df, ROW_COUNT).show(truncate=False)

## Data Preparation - Handle Missing Values

In [9]:
df = df.transform(prep.remove_missing)
ROW_COUNT = raw_df.count()

## Health Audit - Consistency & Validity

In [ ]:
## Consistency helps us question "Do columns agree with each other logically"?

# 1. timestamp_created should usually be <= timestamp_updated
# 2. playtime_at_review & playtime_last two weeks should be less than lifetime playtime
# 3. all user in the dataset should have at least 1 numbers of review
# 4. author_playtime_forever at later review time should not be lower than earlier review time
format_report(audit.consistency_report(df, ROW_COUNT))

In [ ]:
## Validity helps us question "Do columns within reasonable range"?

# 5. Count-like columns should not be negative
# 6. weighted_vote_score should be between 0 and 1
format_report(audit.validity_report(df, ROW_COUNT))

## Data Preparation - Handle Consistency & Validity

In [ ]:
df = (
    # Inconsistent data
    df.transform(prep.remove_timestamp_consistency)
        .transform(prep.remove_playtime_consistency)
        .transform(prep.remove_playtime_forever_decreases_over_time)
    # Invalid data
        .transform(prep.remove_negative_count_values)
        .transform(prep.remove_invalid_weighted_vote_score)
        .transform(prep.remove_invalid_timestamps)
)
ROW_COUNT = raw_df.count()

## Health Audit - Anomaly & Outliers

In [ ]:
## Anomaly indicates suspicious value, not automatically wrong
ano_report = audit.anomaly_report(df, ROW_COUNT)

# 1. Check for outlier value using summary statistics
display(ano_report.get('numeric_describe_df').toPandas())

# 2. Create report regarding anomaly report for suspicious features (1) votes_funny artifact
# 3. Check for instances where votes_funny reached max value
# 4. Check for instances where votes_funny near max value
format_report(ano_report)

# 5. Find the largest vote_funny value that follows right after ARTIFACT_THRESHOLD, compute its propoertion comparing to max value
# If it's between 60% - 95%, it will indicate broader range of artifact from API and we will need to dive deeper
print_section("Largest non-artifact votes_funny value below threshold")
ano_value, ano_percentage = ano_report.get('vote_max_non_artifact', (None, None))
if ano_value is not None: print(ano_value)
if ano_percentage is not None: print(ano_percentage)
del ano_report, ano_value, ano_percentage

## Data Preparation - Handle Abnormal Data

In [ ]:
df = (
    df.transform(prep.remove_impossible_two_week_playtime)
        .transform(prep.remove_impossible_vote_funny)
)
ROW_COUNT = raw_df.count()

## Health Audit - Noise

In [ ]:
## Noises are values that are valid but possibly not useful & unstable

# 1. Check for columns with mostly zeros
audit.noise_report(df, ROW_COUNT).show(truncate=False)

## Health Audit - Uniqueness & Duplicates

In [ ]:
## Uniqueness of catgorical features tells us insight of categories, while for numerical features,
## it indicates potential few_unique_values_per_column issue or possibility for discretization to a categorical type

# 1. Numbers of unique values for categorical/numerical & ID columns
audit.uniqueness_report(df, ROW_COUNT).show(truncate=False)

## Duplicate tells us how many rows has exact duplicates and how frequent would a user to review the same game

# 2. Numbers of duplicate found in dataframe
# 3. Does the same user appear to have created multiple review records for the same game at the same time?
# 4. Inspect more details ("author_steamid", "appid", "timestamp_created") regarding instances in #3
# 5. Print out the duplicate reviews
dup_reports = audit.duplicate_report(df, ROW_COUNT)
for (report, df) in zip(dup_reports['summary_rows'], dup_reports['issue_dfs'].values()):
    print_section(report['issue_name'])
    for name, val in report.items():
        if name == 'issue_name': continue
        print(f"{name}: {val}")
    df.show(n=5, truncate=False)

# 6. Create report on metadata difference on same (author ID, game ID, review date created)
# UN-ACHIEVE-ABLE; reason: too much join, putting too pressure on spark worker memory

## Data Preparation - Handle Duplicates

In [ ]:
df = (
    df.transform(prep.remove_duplicate_rows)
        .transform(prep.remove_duplicate_reviews)
)

## Write Cleaned Full Data to Parquet File

In [ ]:
write_spark_parquet(df=df, path=paths.cleaned_parquet)

In [ ]:
spark.stop()